In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [7]:


load_dotenv(override=True)

openrouter_api = os.getenv("OPENROUTER_API_KEY")
groq_api = os.getenv("GROQ_API_KEY")
if openrouter_api:
    display("Openrouter api found")
else:
    display("OPENROUTER NOT FOUND")
if groq_api:
    display("groq api found")
else:
    display("Groq api not found")

groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key= groq_api)

openrouter = OpenAI(base_url="https://openrouter.ai/api/v1" , api_key= openrouter_api)

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key= "ollama")

groq_model ="llama-3.3-70b-versatile"
openrouter_model = "openai/gpt-oss-120b:free"
ollama_model = "llama3.1:8b"


'Openrouter api found'

'groq api found'

In [8]:
system_message = """
You are a helpful assistant for an Airline called Nepal Airlines.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [12]:



def chat(message,history):
    history = [{'role':h["role"],'content':h['content']} for h in history]
    messages = [{'role':'system','content':system_message}] + history + [{'role':'user','content':message}]
    response = ollama.chat.completions.create(
        model = ollama_model,
        messages = messages
    )
    return response.choices[0].message.content
gr.ChatInterface(fn = chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


# TOOLS

#### with tools, you can write a function and have the LLM call that function as a part of its response

In [84]:
system_message = """You are a helpful airline ticket booking assistant. 
You help customers find ticket prices to different cities.

RULES:
- You have access to a tool called get_ticket_price
- ALWAYS use the tool to get prices, never guess or make up prices
- NEVER show raw JSON or tool call syntax to the user
- Respond naturally and conversationally
- If asked about other cities, say they are not available
"""

In [42]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool call for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown tickcet price")
    return f"The price of a ticket to {destination_city} is {price}"

In [43]:
get_ticket_price("BERLIN")

Tool call for city BERLIN


'The price of a ticket to BERLIN is $499'

### JSON STRUCTURE SO THAT LLM CAN UNDERSTAND THE FUNCTION

In [44]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of an airlines ticket to the desination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city":{
                "type":"string",
                "description":"The city that the customer wants to travel to",
            },
        },
        "required": ['destination_city'],
        'additionalProperties': False
    }
}

In [45]:
tools = [{'type':'function', "function": price_function}]

In [46]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of an airlines ticket to the desination city',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

### getting LLM to use our tools

In [47]:
# Step 1: What is message?
# When the AI wants to call a tool, instead of replying with text, it sends back a special message object that looks like this internally:
# message = {
#     "role": "assistant",
#     "tool_calls": [
#         {
#             "id": "call_abc123",
#             "function": {
#                 "name": "get_ticket_price",
#                 "arguments": '{"destination_city": "Tokyo"}'
#             }
#         }
#     ]
# }
# tool_call = message.tool_calls[0]
# message.tool_calls is a list because the AI could request multiple tools at once.
# [0] just grabs the first one.

In [48]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [49]:
def chatt(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history] 
    messages = [{"role":"system", "content": system_message}]+history+[{"role":"user","content":message}]
    response = ollama.chat.completions.create(
        model = ollama_model,
        messages = messages,
        tools = tools
    )
    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = ollama.chat.completions.create(
            model = ollama_model,
            messages = messages,
            tools=tools
        )
    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(fn = chatt).launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


Tool call for city Paris
Tool call for city Paris
Tool call for city Berlin
Tool call for city Tokyo


## MULTIPLE TOOL CALLS

In [57]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [62]:
def chatt(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history] 
    messages = [{"role":"system", "content": system_message}]+history+[{"role":"user","content":message}]
    response = ollama.chat.completions.create(
        model = ollama_model,
        messages = messages,
        tools = tools
    )
    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        response = handle_tool_calls(assistant_message)
        messages.append({
            "role": "assistant",
            "content": assistant_message.content,
            "tool_calls": assistant_message.tool_calls
        })
        messages.extend(response)
        response = ollama.chat.completions.create(
            model = ollama_model,
            messages = messages,
            tools=tools
        )
    return response.choices[0].message.content


In [72]:
# FIXED FUNCTION

def chatt(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = groq.chat.completions.create(
        model=groq_model,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_responses = handle_tool_calls(assistant_message)

        # ✅ Convert tool_calls Pydantic objects → plain dicts
        messages.append({
            "role": "assistant",
            "content": assistant_message.content or "",   # ✅ None → ""
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                }
                for tc in assistant_message.tool_calls    # ✅ serialized properly
            ]
        })

        messages.extend(tool_responses)                   # ✅ extend not append

        response = groq.chat.completions.create(
            model=groq_model,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

In [73]:
gr.ChatInterface(fn=chatt).launch()

* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


Tool call for city Berlin
Tool call for city Berlin
Tool call for city Paris
Tool call for city London
Tool call for city Paris
Tool call for city London


# WITH SQLite

In [74]:
import sqlite3

In [75]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [76]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [77]:
get_ticket_price("london")

DATABASE TOOL CALLED: Getting price for london


'No price data available for this city'

In [78]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [79]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [80]:
get_ticket_price("tokyo")

DATABASE TOOL CALLED: Getting price for tokyo


'Ticket price to tokyo is $1420.0'

In [81]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [82]:
# FIXED FUNCTION

def chatt(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = groq.chat.completions.create(
        model=groq_model,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_responses = handle_tool_calls(assistant_message)

        # ✅ Convert tool_calls Pydantic objects → plain dicts
        messages.append({
            "role": "assistant",
            "content": assistant_message.content or "",   # ✅ None → ""
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                }
                for tc in assistant_message.tool_calls    # ✅ serialized properly
            ]
        })

        messages.extend(tool_responses)                   # ✅ extend not append

        response = groq.chat.completions.create(
            model=groq_model,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

In [85]:
gr.ChatInterface(fn=chatt).launch(share=True)

* Running on local URL:  http://127.0.0.1:7884
* Running on public URL: https://a24e1ee45fae165196.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


DATABASE TOOL CALLED: Getting price for Sydney
DATABASE TOOL CALLED: Getting price for Tokyo
